# Thực nghiệm 2: In-domain metric depth trên NYUv2

**Depth Anything V1, CVPR 2024, Table 3 — 654 ảnh test.**
Đánh giá checkpoint indoor đã fine-tune; không huấn luyện lại.

| Chỉ số | δ₁ ↑ | δ₂ ↑ | δ₃ ↑ | AbsRel ↓ | RMSE ↓ | log10 ↓ |
|---|---:|---:|---:|---:|---:|---:|
| Paper | **0.984** | **0.998** | **1.000** | **0.056** | **0.206 m** | **0.024** |

Prediction theo mét; **không căn chỉnh scale/shift với GT**.
[Protocol và phạm vi các bảng NYUv2](docs/table3/nyuv2-experiments.md).


## 1. Môi trường
Chọn kernel `.venv` của repository; chạy notebook từ thư mục gốc.

In [1]:
import sys
import torch
import pandas as pd
from evaluate_nyuv2_metric import evaluate_nyuv2_metric, PAPER

print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Python: D:\Code\Master\Depth-Anything\.venv\Scripts\python.exe
PyTorch: 2.4.0+cu121


CUDA: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 2. Đánh giá đủ test split

Đọc `rawDepths`, lượng tử hóa 1 mm và chuẩn bị ảnh JPEG/viền đen như BTS; mask Eigen, input resize 392×518 trong model,
flip augmentation theo mã chính thức. Cache giúp chạy lại không cần inference mới.
Khác thực nghiệm relative, ở đây dùng raw depth đã lượng tử hóa và mask Eigen; metric tính trực tiếp ở độ sâu mét, không phải disparity.


In [2]:
summary = evaluate_nyuv2_metric()
assert summary['full_test_split'] and summary['num_images'] == 654


NYUv2 metric indoor:   0%|          | 0/654 [00:00<?, ?it/s]

NYUv2 metric indoor:  53%|█████▎    | 347/654 [00:10<00:08, 34.69it/s]

NYUv2 metric indoor: 100%|██████████| 654/654 [00:17<00:00, 37.54it/s]

{
  "created_at": "2026-11-26T15:07:22.362674+00:00",
  "experiment": "in-domain metric depth, evaluation of released NYUv2-finetuned checkpoint",
  "paper_table": 3,
  "num_images": 654,
  "full_test_split": true,
  "metrics": {
    "delta1": 0.9835544375390471,
    "delta2": 0.9980002137923506,
    "delta3": 0.99961077810392,
    "abs_rel": 0.05597970240239612,
    "rmse": 0.20614259894538545,
    "log10": 0.02438664402163357
  },
  "paper_reference": {
    "delta1": 0.984,
    "delta2": 0.998,
    "delta3": 1.0,
    "abs_rel": 0.056,
    "rmse": 0.206,
    "log10": 0.024
  },
  "difference_from_paper": {
    "delta1": -0.0004455624609528419,
    "delta2": 2.1379235060070556e-07,
    "delta3": -0.00038922189608003865,
    "abs_rel": -2.029759760387856e-05,
    "rmse": 0.00014259894538545814,
    "log10": 0.0003866440216335684
  },
  "protocol": {
    "checkpoint_sha256": "e25a96605a2ea2dbbe57308d024e89a803fa87fe981bd51d4cee245f1de57977",
    "encoder_checkpoint_sha256": "6fda5199f004

## 3. So sánh sáu chỉ số với Table 3
Các giá trị measured lấy từ lượt chạy; cột paper chỉ là tham chiếu.

In [3]:
comparison = pd.DataFrame([
    {'Metric': name, 'Our result': summary['metrics'][name],
     'Paper Table 3': reference,
     'Difference': summary['metrics'][name] - reference}
    for name, reference in PAPER.items()
])
display(comparison)
print('Full test split:', summary['full_test_split'], '| Images:', summary['num_images'])
print('Training performed:', summary['training_performed'])


,Metric,Our result,Paper Table 3,Difference
0,delta1,0.983554,0.984,-4.455625e-04
1,delta2,0.998000,0.998,2.137924e-07
2,delta3,0.999611,1.000,-3.892219e-04
3,abs_rel,0.055980,0.056,-2.029760e-05
4,rmse,0.206143,0.206,1.425989e-04
5,log10,0.024387,0.024,3.866440e-04


Full test split: True | Images: 654
Training performed: False


## 4. Các thực nghiệm liên quan còn lại

- Table 10/11, nhánh **Ours**, dùng lại kết quả metric này.
- Các baseline MiDaS/DINOv2 fine-tuned và ablation Table 6/9 cần checkpoint riêng;
  chưa có trong workspace, không được coi là đã tái hiện.
- Table 5 dùng NYUv2 làm **nguồn** fine-tune, test ở SUN RGB-D/iBims-1/HyperSim.
- Tự fine-tune cần tập train 24.231 cặp RGB/depth theo file train của repo;
  795 ảnh train trong labeled MAT không thay thế được tập đó.
